# SRISK Calculation - Interactive Debug Notebook

This notebook contains the main SRISK calculation workflow from `scripts/04_calculate_srisk.py`.
Run cells step-by-step to debug and inspect intermediate results.

**Pipeline:**
1. Load processed data (aligned debt + equity)
2. Load LRMES estimates
3. Calculate SRISK
4. System aggregates
5. Summary statistics
6. Country aggregates
7. Decomposition analysis

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_dataframe, save_dataframe
from src.models.srisk import (
    calculate_srisk_multi,
    calculate_system_srisk,
    calculate_srisk_contribution,
    srisk_summary_statistics,
    calculate_srisk_by_country,
    decompose_srisk
)
from src.utils.config import CONFIG, PROCESSED_DATA_DIR, RESULTS_DATA_DIR
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ Imports successful")
print(f"\nConfiguration:")
print(f"  Capital ratio: {CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%")
print(f"  Crisis threshold: {CONFIG.CRISIS_THRESHOLD*100:.1f}%")
print(f"  Horizon: {CONFIG.CRISIS_HORIZON_WEEKS} weeks")

## Step 1: Load Data

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load aligned data
aligned_data = load_dataframe("aligned_daily_data", PROCESSED_DATA_DIR)
print(f"✓ Aligned data: {aligned_data.shape}")
print(f"  Date range: {aligned_data.index.min()} to {aligned_data.index.max()}")

# Load LRMES estimates
lrmes_df = load_dataframe("lrmes_estimates", RESULTS_DATA_DIR)
print(f"\n✓ LRMES estimates: {len(lrmes_df)} banks")

# Load bank universe
bank_universe = load_dataframe("banks_universe_clean", PROCESSED_DATA_DIR)
print(f"✓ Bank universe: {len(bank_universe)} banks")

In [ ]:
# Inspect LRMES estimates
print("LRMES Estimates (top 10 by LRMES):")
display(lrmes_df.sort_values('lrmes', ascending=False).head(10))

In [ ]:
# Convert LRMES to time series (broadcast to all dates)
print("Converting LRMES to time series...")

lrmes_values = dict(zip(lrmes_df["bank_ric"], lrmes_df["lrmes"]))

# Create time series DataFrame
lrmes_ts = pd.DataFrame(
    {inst: [lrmes_values[inst]] * len(aligned_data) for inst in lrmes_values.keys()},
    index=aligned_data.index
)

print(f"✓ LRMES time series: {lrmes_ts.shape}")
print(f"  Columns: {list(lrmes_ts.columns[:5])} ...")

## Step 2: Calculate SRISK

In [ ]:
print("=" * 60)
print("CALCULATING SRISK")
print("=" * 60)
print(f"Capital ratio: {CONFIG.CAPITAL_RATIO_BASEL * 100:.1f}%")
print()

# Calculate SRISK
srisk = calculate_srisk_multi(
    aligned_data=aligned_data,
    lrmes_df=lrmes_ts,
    bank_universe=bank_universe,
    k=CONFIG.CAPITAL_RATIO_BASEL
)

print(f"✓ SRISK calculated for {len(srisk.columns)} banks")
print(f"  Shape: {srisk.shape}")

In [ ]:
# Inspect SRISK - latest values
print("Latest SRISK (top 10 banks):")
latest_srisk = srisk.iloc[-1].sort_values(ascending=False)
print(latest_srisk.head(10).apply(lambda x: f"${x/1e9:.2f}B"))

In [ ]:
# Save SRISK time series
save_dataframe(srisk, "srisk_timeseries", RESULTS_DATA_DIR, {
    "description": f"SRISK time series (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL,
    "crisis_threshold": CONFIG.CRISIS_THRESHOLD,
    "horizon_weeks": CONFIG.CRISIS_HORIZON_WEEKS
})
print("✓ Saved: srisk_timeseries.parquet")

## Step 3: System Aggregates

In [ ]:
print("=" * 60)
print("SYSTEM-WIDE AGGREGATION")
print("=" * 60)

# System SRISK (sum of positive SRISK)
system_srisk = calculate_system_srisk(srisk)

print(f"\nSystem SRISK (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%):")
print(f"  Latest: ${system_srisk.iloc[-1] / 1e9:.2f}B")
print(f"  Mean: ${system_srisk.mean() / 1e9:.2f}B")
print(f"  Max: ${system_srisk.max() / 1e9:.2f}B")
print(f"  Min: ${system_srisk.min() / 1e9:.2f}B")

In [ ]:
# Plot system SRISK over time
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(system_srisk.index, system_srisk / 1e9)
plt.title('System-wide SRISK Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('SRISK ($ Billions)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate contributions (percentage of system SRISK)
contributions = calculate_srisk_contribution(srisk)

print(f"\nTop 5 contributors (latest):")
latest_contrib = contributions.iloc[-1].sort_values(ascending=False).head()
for bank, pct in latest_contrib.items():
    print(f"  - {bank}: {pct:.2f}%")

In [ ]:
# Save system aggregates
save_dataframe(system_srisk, "system_srisk", RESULTS_DATA_DIR, {
    "description": "System-wide SRISK (sum of positive SRISK)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

save_dataframe(contributions, "srisk_contributions", RESULTS_DATA_DIR, {
    "description": "Bank contributions to system SRISK (%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: system_srisk.parquet")
print("✓ Saved: srisk_contributions.parquet")

## Step 4: Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

# Summary statistics
summary = srisk_summary_statistics(srisk, bank_universe)

print(f"\nTop 10 banks by mean SRISK:")
display(summary.head(10)[["bank_ric", "country", "mean_srisk", "latest_srisk", "max_srisk"]])

In [ ]:
# Save summary
save_dataframe(summary, "srisk_summary", RESULTS_DATA_DIR, {
    "description": f"SRISK summary statistics (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_summary.parquet")

## Step 5: Country Aggregates

In [ ]:
print("=" * 60)
print("COUNTRY AGGREGATION")
print("=" * 60)

country_srisk = calculate_srisk_by_country(srisk, bank_universe)

print(f"\nSRISK by country (latest):")
latest_country = country_srisk.iloc[-1].sort_values(ascending=False)
for country, srisk_val in latest_country.items():
    print(f"  - {country}: ${srisk_val / 1e9:.2f}B")

In [ ]:
# Plot country SRISK
plt.figure(figsize=(10, 6))
latest_country.plot(kind='barh')
plt.title('SRISK by Country (Latest)', fontsize=14, fontweight='bold')
plt.xlabel('SRISK ($ Billions)')
plt.ylabel('Country')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Save country aggregates
save_dataframe(country_srisk, "srisk_by_country", RESULTS_DATA_DIR, {
    "description": f"SRISK aggregated by country (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_by_country.parquet")

## Step 6: Decomposition Analysis

In [ ]:
print("=" * 60)
print("DECOMPOSITION ANALYSIS")
print("=" * 60)
print("Analyzing top 5 banks by latest SRISK...")

# Identify top banks
top_banks = srisk.iloc[-1].sort_values(ascending=False).head(5).index.tolist()
print(f"\nTop 5 banks: {top_banks}")

In [ ]:
# Decompose SRISK for each top bank
decomp_results = []

for bank_ric in top_banks:
    debt_col = f"fund_{bank_ric}_TotLiab"
    equity_col = f"mktcap_{bank_ric}"
    
    if debt_col in aligned_data.columns and equity_col in aligned_data.columns:
        debt = aligned_data[debt_col]
        equity = aligned_data[equity_col]
        lrmes = lrmes_ts[bank_ric]
        
        decomp = decompose_srisk(debt, equity, lrmes, k=CONFIG.CAPITAL_RATIO_BASEL)
        
        # Summarize
        decomp_results.append({
            "bank_ric": bank_ric,
            "avg_size_effect": decomp["size_effect"].mean(),
            "avg_leverage_effect": decomp["leverage_effect"].mean(),
            "avg_risk_effect": decomp["risk_effect"].mean(),
            "total_change": decomp["srisk"].iloc[-1] - decomp["srisk"].iloc[0]
        })

decomp_df = pd.DataFrame(decomp_results)
print(f"\nDecomposition (average daily effects):")
display(decomp_df)

In [ ]:
# Save decomposition
save_dataframe(decomp_df, "srisk_decomposition", RESULTS_DATA_DIR, {
    "description": "SRISK decomposition (size, leverage, risk effects)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_decomposition.parquet")

## Final Summary

In [ ]:
print("=" * 60)
print("✓ SRISK CALCULATION COMPLETED")
print("=" * 60)
print(f"\nCapital ratio used: {CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%")
print(f"Crisis threshold: {CONFIG.CRISIS_THRESHOLD*100:.1f}% market decline")
print(f"Horizon: {CONFIG.CRISIS_HORIZON_WEEKS} weeks")
print(f"\nResults saved to: {RESULTS_DATA_DIR}")
print("\nKey outputs:")
print("  - srisk_timeseries.parquet - SRISK time series")
print("  - system_srisk.parquet - System-wide SRISK")
print("  - srisk_contributions.parquet - Bank contributions (%)")
print("  - srisk_summary.parquet - Summary statistics")
print("  - srisk_by_country.parquet - Country aggregates")
print("  - srisk_decomposition.parquet - Decomposition analysis")